# 24 Final Conclusion And Best Model

This notebook is the short thesis wrap-up after notebook `23`.

Notebook `23_fs3_decision_relevant_forecast_evaluation.ipynb` is the detailed decision layer. This notebook only restates the final DA recommendation and the thesis next-step decision in a compact form.


In [1]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.models import naive_model_names
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


def ensure_required_modules(module_names: list[str], install_command: str | None = None) -> None:
    missing = [module_name for module_name in module_names if importlib.util.find_spec(module_name) is None]
    if not missing:
        return

    message_lines = [
        "Missing required package(s) in the active notebook interpreter: " + ", ".join(missing),
        f"Active interpreter: {sys.executable}",
    ]
    if install_command:
        message_lines.append(f"Install command: {install_command}")
    raise RuntimeError("\n".join(message_lines))


def run_command_with_live_output(command: list[str]) -> None:
    print("Running command:")
    print(" ".join(str(part) for part in command))
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    output_tail: list[str] = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        output_tail.append(line)
        if len(output_tail) > 40:
            output_tail.pop(0)

    return_code = process.wait()
    if return_code != 0:
        tail_text = "".join(output_tail).strip()
        message = f"Command failed with exit code {return_code}."
        if tail_text:
            message += "\nLast output:\n" + tail_text
        raise RuntimeError(message)


In [2]:
from hourly_da.core.forecast_evaluation import (
    build_model_selection_summary,
    build_recommendation_inputs,
    compute_classical_metrics,
    compute_daily_level_metrics,
    compute_horizon_metrics,
    default_decision_thresholds,
    discover_final_candidate_runs,
    load_candidate_predictions,
    load_candidate_runtime_summary,
    load_official_naive_denominators,
    recommend_candidate,
)


In [3]:
DECISION_THRESHOLDS = default_decision_thresholds()
comparison_discovery = discover_final_candidate_runs(output_root)
candidate_frame = comparison_discovery["candidates"].copy()
candidate_predictions = load_candidate_predictions(candidate_frame=candidate_frame, config=config)
candidate_runtime = load_candidate_runtime_summary(candidate_frame)
naive_denominators = load_official_naive_denominators(
    run_dir=comparison_discovery["rmae_reference_run_dir"],
    official_naive_model=str(comparison_discovery["official_naive_model"]),
)
classical_metrics = compute_classical_metrics(candidate_predictions, naive_denominators)
horizon_metrics = compute_horizon_metrics(candidate_predictions, naive_denominators)
_, daily_level_summary = compute_daily_level_metrics(candidate_predictions)
selection_inputs = build_recommendation_inputs(
    classical_metrics,
    horizon_metrics,
    daily_level_summary,
    pd.DataFrame(),
    pd.DataFrame(),
    pd.DataFrame(),
    candidate_runtime,
)
recommendation = recommend_candidate(selection_inputs, thresholds=DECISION_THRESHOLDS)
selection_summary = build_model_selection_summary(selection_inputs, recommendation)

display(selection_summary)


,candidate_context,candidate_label,classical_metrics_summary,horizon_robustness_summary,level_retention_summary,shape_retention_summary,expensive_hour_recall_summary,cheap_hour_recall_summary,tail_stress_note,runtime_note,implementability_note,recommended_yes_no,thesis_use_limitations,notes
0,fs3_pruned_candidate,XGBoost FS3 pruned candidate,"val rMAE 0.854, val MAE 23.78, test rMAE 0.886",100% of validation horizons below rMAE 1.00,"mean daily level MAE 11.99, mean daily bias -0.06",Unavailable,Unavailable,Unavailable,Unavailable,"fit 0.35s, predict 1.64s",Reduced FS3 tree candidate. Still less interpr...,True,Document deterministic-only limitations and re...,XGBoost FS3 pruned candidate improves validati...
1,fs2_reference,Best FS2 reference,"val rMAE 0.894, val MAE 24.90, test rMAE 0.893",100% of validation horizons below rMAE 1.00,"mean daily level MAE 14.40, mean daily bias +4.03",Unavailable,Unavailable,Unavailable,Unavailable,"fit 0.16s, predict 1.08s","Operationally practical nonlinear model, but l...",False,Document deterministic-only limitations and re...,
2,fs3_promoted,LEAR FS3 promoted,"val rMAE 1.011, val MAE 28.16, test rMAE 1.113",20% of validation horizons below rMAE 1.00,"mean daily level MAE 11.99, mean daily bias -0.54",Unavailable,Unavailable,Unavailable,Unavailable,"fit 0.19s, predict 1.45s",Full FS3 causal stack on a linear base. More e...,False,Document deterministic-only limitations and re...,
3,fs3_pruned_candidate,LEAR FS3 pruned candidate,"val rMAE 1.025, val MAE 28.54, test rMAE 1.037",20% of validation horizons below rMAE 1.00,"mean daily level MAE 11.45, mean daily bias -0.67",Unavailable,Unavailable,Unavailable,Unavailable,"fit 0.12s, predict 1.18s",Reduced FS3 linear candidate. Keeps some FS3 b...,False,Document deterministic-only limitations and re...,
4,benchmark,Naive previous week,"val rMAE 1.000, val MAE 27.86, test rMAE 1.000",0% of validation horizons below rMAE 1.00,"mean daily level MAE 22.43, mean daily bias +0.36",Unavailable,Unavailable,Unavailable,Unavailable,"fit 0.00s, predict 0.00s",Minimal implementation cost and maximum transp...,False,Baseline only; use only if stronger models fai...,
5,fs3_promoted,XGBoost FS3 promoted,"val rMAE 0.904, val MAE 25.20, test rMAE 0.860",100% of validation horizons below rMAE 1.00,"mean daily level MAE 12.72, mean daily bias +2.89",Unavailable,Unavailable,Unavailable,Unavailable,"fit 1.03s, predict 2.89s",Highest complexity among current candidates be...,False,Document deterministic-only limitations and re...,


In [4]:
recommended_key = recommendation.get("recommended_candidate_key")
recommended_label = recommendation.get("recommended_candidate_label")
recommended_row = selection_inputs[selection_inputs["candidate_key"] == recommended_key].copy()
recommended_row = recommended_row.iloc[0] if not recommended_row.empty else None
best_fs2_row = selection_inputs[selection_inputs["candidate_key"] == "best_fs2_reference"].copy()
best_fs2_row = best_fs2_row.iloc[0] if not best_fs2_row.empty else None
naive_row = selection_inputs[selection_inputs["candidate_key"] == "official_naive_benchmark"].copy()
naive_row = naive_row.iloc[0] if not naive_row.empty else None

if recommended_row is None:
    display(Markdown("No final DA candidate could be recommended from the available artifacts."))
else:
    delta_vs_fs2 = (
        float(best_fs2_row["rmae"] - recommended_row["rmae"])
        if best_fs2_row is not None and pd.notna(best_fs2_row["rmae"]) and pd.notna(recommended_row["rmae"])
        else float("nan")
    )
    delta_vs_naive = (
        float(naive_row["rmae"] - recommended_row["rmae"])
        if naive_row is not None and pd.notna(naive_row["rmae"]) and pd.notna(recommended_row["rmae"])
        else float("nan")
    )
    conclusion_lines = [
        "### Final DA recommendation",
        f"- Recommended candidate: `{recommended_label}`.",
        f"- Validation `rMAE` vs official naive: `{float(recommended_row['rmae']):.3f}`.",
        f"- Test `rMAE` vs official naive: `{float(recommended_row['test_rmae']):.3f}`." if pd.notna(recommended_row["test_rmae"]) else "- Test `rMAE` vs official naive: unavailable.",
        f"- Validation improvement vs best/current FS2 reference: `{delta_vs_fs2:.3f}` rMAE points." if pd.notna(delta_vs_fs2) else "- Validation improvement vs best/current FS2 reference: unavailable.",
        f"- Validation improvement vs official naive benchmark: `{delta_vs_naive:.3f}` rMAE points." if pd.notna(delta_vs_naive) else "- Validation improvement vs official naive benchmark: unavailable.",
        f"- Decision reason: {recommendation.get('decision_reason')}",
        f"- DA forecasting should {'stop here' if recommendation.get('stop_da_here') else 'continue'} before further thesis expansion.",
        f"- Moving on to mFRR forecasting is {'reasonable' if recommendation.get('move_to_mfrr') else 'not yet reasonable'} under the current evidence.",
        "- Detailed robustness, level, shape, top-k, tail, and objective-week evidence lives in notebook `23_fs3_decision_relevant_forecast_evaluation.ipynb`.",
    ]
    display(Markdown("\n".join(conclusion_lines)))


### Final DA recommendation
- Recommended candidate: `XGBoost FS3 pruned candidate`.
- Validation `rMAE` vs official naive: `0.854`.
- Test `rMAE` vs official naive: `0.886`.
- Validation improvement vs best/current FS2 reference: `0.040` rMAE points.
- Validation improvement vs official naive benchmark: `0.146` rMAE points.
- Decision reason: XGBoost FS3 pruned candidate improves validation rMAE enough over the best FS2 reference without triggering the configured simplicity fallback.
- DA forecasting should stop here before further thesis expansion.
- Moving on to mFRR forecasting is reasonable under the current evidence.
- Detailed robustness, level, shape, top-k, tail, and objective-week evidence lives in notebook `23_fs3_decision_relevant_forecast_evaluation.ipynb`.